PREMIER PASSAGE
01 → 20
       ↓
validation_enseignant_ressources.csv
       ↓
ENSEIGNANT

DEUXIÈME PASSAGE
21 → 24
       ↓
détection besoins non couverts
       ↓
25A → prompts LLM
25B → génération LLM
       ↓
validation_enseignant_llm.csv
       ↓
ENSEIGNANT

TROISIÈME PASSAGE
rechargement validation LLM
       ↓
consolidation
       ↓
ressources_finales.csv
       ↓
ressources_llm_validees_pour_knowledge.csv

In [1]:
# ============================================================
# CELLULE 1 — Imports
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


RANDOM_STATE = 42


print("Imports chargés.")

Imports chargés.


In [2]:
# ============================================================
# CELLULE 2 — Répertoires
# ============================================================

def trouver_racine_projet():

    candidats = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]

    for racine in candidats:

        if (
            racine
            / "outputs"
            / "05_groupes_recommandation"
        ).exists():

            return racine.resolve()

    raise FileNotFoundError(
        "outputs/05_groupes_recommandation introuvable."
    )


PROJECT_ROOT = trouver_racine_projet()


INPUT_05 = (
    PROJECT_ROOT
    / "outputs"
    / "05_groupes_recommandation"
    / "tables"
)


OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "06_RAG_LLM_HITL"
)


TABLE_DIR = OUTPUT_DIR / "tables"
RAG_DIR = OUTPUT_DIR / "rag"
HITL_DIR = OUTPUT_DIR / "hitl"
LLM_DIR = OUTPUT_DIR / "llm"


for dossier in [
    OUTPUT_DIR,
    TABLE_DIR,
    RAG_DIR,
    HITL_DIR,
    LLM_DIR
]:

    dossier.mkdir(
        parents=True,
        exist_ok=True
    )


print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT 05     :", INPUT_05)
print("OUTPUT 06    :", OUTPUT_DIR)

PROJECT_ROOT : /home/andrycomputer/Documents/Model-eval-pers-elearning
INPUT 05     : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/05_groupes_recommandation/tables
OUTPUT 06    : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/06_RAG_LLM_HITL


Cellule 3 — Charger les groupes du Notebook 05

In [3]:
# ============================================================
# CELLULE 3 — Chargement des groupes
# ============================================================

RECOMMANDATIONS_GROUPES = pd.read_csv(
    INPUT_05
    / "recommandations_contenu_groupes.csv"
)


BASE_GROUPES = pd.read_csv(
    INPUT_05
    / "base_groupes_pedagogiques.csv"
)


colonnes_groupes_requises = {
    "ID_GROUPE",
    "PARCOURS",
    "PROGRAMME_COMMUN",
    "NIVEAU",
    "LACUNE_PRINCIPALE"
}


colonnes_manquantes = (
    colonnes_groupes_requises
    -
    set(RECOMMANDATIONS_GROUPES.columns)
)


assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes : "
    f"{colonnes_manquantes}"
)


assert (
    RECOMMANDATIONS_GROUPES[
        "ID_GROUPE"
    ]
    .is_unique
)


print(
    "Recommandations groupes :",
    RECOMMANDATIONS_GROUPES.shape
)

print(
    "Base apprenants :",
    BASE_GROUPES.shape
)


display(
    RECOMMANDATIONS_GROUPES[
        [
            "ID_GROUPE",
            "PARCOURS",
            "PROGRAMME_COMMUN",
            "NIVEAU",
            "LACUNE_PRINCIPALE"
        ]
    ]
)

Recommandations groupes : (12, 16)
Base apprenants : (125, 77)


,ID_GROUPE,PARCOURS,PROGRAMME_COMMUN,NIVEAU,LACUNE_PRINCIPALE
0,G001,BI,PROGRAMME_BI,Avance,Modèle en étoile | Relations Power BI | Cardin...
1,G002,BI,PROGRAMME_BI,Debutant,Typage des clés | Nettoyage de cellules fusion...
2,G003,BI,PROGRAMME_BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...
3,G004,DA,PROGRAMME_DA,Avance,Power Query (évaluation paresseuse) | Power Pi...
4,G005,DA,PROGRAMME_DA,Debutant,Power Query (fusion)
5,G006,DA,PROGRAMME_DA,Intermediaire,Power Pivot (schéma en étoile) | Power Query (...
6,G007,DS,PROGRAMME_DS,Avance,Machine Learning (déséquilibre) | Pandas (join...
7,G008,DS,PROGRAMME_DS,Debutant,Machine Learning (généralisation) | Pandas (jo...
8,G009,DS,PROGRAMME_DS,Intermediaire,Machine Learning (déséquilibre) | Statistiques...
9,G010,IA,PROGRAMME_IA,Avance,Open WebUI (Skills) | LLM (paramètres) | Open ...


Cellule 4 — Emplacement du catalogue

In [4]:
# ============================================================
# CELLULE 4 — Répertoire des ressources pédagogiques
# ============================================================

RESOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "ressources_pedagogiques"
)


RESOURCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESOURCE_CSV = (
    RESOURCE_DIR
    / "course_resources.csv"
)


print(
    "Répertoire ressources :",
    RESOURCE_DIR
)

print(
    "Catalogue attendu :",
    RESOURCE_CSV
)

Répertoire ressources : /home/andrycomputer/Documents/Model-eval-pers-elearning/data/ressources_pedagogiques
Catalogue attendu : /home/andrycomputer/Documents/Model-eval-pers-elearning/data/ressources_pedagogiques/course_resources.csv


Cellule 5 — Schéma du catalogue

In [5]:
# ============================================================
# CELLULE 5 — Schéma du catalogue
# ============================================================

COLONNES_REQUISES = [
    "RESOURCE_ID",
    "PARCOURS",
    "CHAPITRE",
    "LECON",
    "TITRE",
    "TYPE_RESSOURCE",
    "SOURCE"
]


COLONNES_RECOMMANDEES = [
    "COMPETENCE",
    "MOTS_CLES"
]


print(
    "Colonnes obligatoires :",
    COLONNES_REQUISES
)

print(
    "Colonnes recommandées :",
    COLONNES_RECOMMANDEES
)

Colonnes obligatoires : ['RESOURCE_ID', 'PARCOURS', 'CHAPITRE', 'LECON', 'TITRE', 'TYPE_RESSOURCE', 'SOURCE']
Colonnes recommandées : ['COMPETENCE', 'MOTS_CLES']


Cellule 6 — Charger course_resources.csv

In [6]:
# ============================================================
# CELLULE 6 — Chargement du catalogue
# ============================================================

if not RESOURCE_CSV.exists():

    raise FileNotFoundError(
        f"Fichier introuvable : {RESOURCE_CSV}"
    )


RESSOURCES = pd.read_csv(
    RESOURCE_CSV
)


print(
    "Fichier chargé :",
    RESOURCE_CSV
)

print(
    "Dimensions :",
    RESSOURCES.shape
)

print(
    "Colonnes :",
    RESSOURCES.columns.tolist()
)

Fichier chargé : /home/andrycomputer/Documents/Model-eval-pers-elearning/data/ressources_pedagogiques/course_resources.csv
Dimensions : (168, 8)
Colonnes : ['RESOURCE_ID', 'PARCOURS', 'CHAPITRE', 'LECON', 'TITRE', 'TYPE_RESSOURCE', 'CONTENU', 'SOURCE']


Cellule 7 — Contrôler le catalogue

In [7]:
# ============================================================
# CELLULE 7 — Validation du catalogue
# ============================================================

colonnes_absentes = [
    col
    for col in COLONNES_REQUISES
    if col not in RESSOURCES.columns
]


assert len(colonnes_absentes) == 0, (
    f"Colonnes manquantes : {colonnes_absentes}"
)


assert (
    RESSOURCES[
        "RESOURCE_ID"
    ]
    .notna()
    .all()
)


assert (
    RESSOURCES[
        "RESOURCE_ID"
    ]
    .is_unique
)


PARCOURS_VALIDES = {
    "DA",
    "BI",
    "DS",
    "IA"
}


RESSOURCES[
    "PARCOURS"
] = (
    RESSOURCES[
        "PARCOURS"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


parcours_inconnus = (
    set(
        RESSOURCES[
            "PARCOURS"
        ].unique()
    )
    -
    PARCOURS_VALIDES
)


assert len(parcours_inconnus) == 0, (
    f"Parcours inconnus : {parcours_inconnus}"
)


for col in COLONNES_RECOMMANDEES:

    if col not in RESSOURCES.columns:

        print(
            f"{col} absent du catalogue : "
            "colonne créée vide."
        )

        RESSOURCES[col] = ""


print(
    "Catalogue valide :",
    len(RESSOURCES),
    "ressources."
)

COMPETENCE absent du catalogue : colonne créée vide.
MOTS_CLES absent du catalogue : colonne créée vide.
Catalogue valide : 168 ressources.


Cellule 8 — Distribution du corpus

In [8]:
# ============================================================
# CELLULE 8 — Distribution des ressources
# ============================================================

print("Ressources par parcours")

display(
    RESSOURCES[
        "PARCOURS"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("PARCOURS")
    .reset_index(
        name="N_RESSOURCES"
    )
)


print("Ressources par type")

display(
    RESSOURCES[
        "TYPE_RESSOURCE"
    ]
    .value_counts()
    .rename_axis(
        "TYPE_RESSOURCE"
    )
    .reset_index(
        name="N_RESSOURCES"
    )
)

Ressources par parcours


,PARCOURS,N_RESSOURCES
0,BI,42
1,DA,42
2,DS,42
3,IA,42


Ressources par type


,TYPE_RESSOURCE,N_RESSOURCES
0,lecon,72
1,exercice,72
2,quiz,24


In [9]:
# ============================================================
# CELLULE 9 — Texte d'indexation RAG
# ============================================================

def nettoyer_texte(valeur):

    if pd.isna(valeur):
        return ""

    return str(valeur).strip()


RESSOURCES[
    "TEXTE_RAG"
] = (

    RESSOURCES[
        "TITRE"
    ].apply(nettoyer_texte)

    + " "

    + RESSOURCES[
        "COMPETENCE"
    ].apply(nettoyer_texte)

    + " "

    + RESSOURCES[
        "MOTS_CLES"
    ].apply(nettoyer_texte)

    + " "

    + RESSOURCES[
        "CHAPITRE"
    ].apply(nettoyer_texte)

    + " "

    + RESSOURCES[
        "LECON"
    ].apply(nettoyer_texte)

    + " "

    + RESSOURCES[
        "TYPE_RESSOURCE"
    ].apply(nettoyer_texte)
)


display(
    RESSOURCES[
        [
            "RESOURCE_ID",
            "PARCOURS",
            "CHAPITRE",
            "LECON",
            "TITRE",
            "TYPE_RESSOURCE",
            "TEXTE_RAG"
        ]
    ].head(10)
)

,RESOURCE_ID,PARCOURS,CHAPITRE,LECON,TITRE,TYPE_RESSOURCE,TEXTE_RAG
0,DA-C1-L1-LEC,DA,C1 - Fondamentaux de la donnée,L1.1,Types de données et variables,lecon,Types de données et variables C1 - Fondament...
1,DA-C1-L1-EXO,DA,C1 - Fondamentaux de la donnée,L1.1,Exercice - Types de données et variables,exercice,Exercice - Types de données et variables C1 ...
2,DA-C1-L2-LEC,DA,C1 - Fondamentaux de la donnée,L1.2,"Sources et formats de données (CSV, Excel, JSON)",lecon,"Sources et formats de données (CSV, Excel, JSO..."
3,DA-C1-L2-EXO,DA,C1 - Fondamentaux de la donnée,L1.2,"Exercice - Sources et formats de données (CSV,...",exercice,"Exercice - Sources et formats de données (CSV,..."
4,DA-C1-L3-LEC,DA,C1 - Fondamentaux de la donnée,L1.3,Cycle de vie de la donnée et rôle du Data Analyst,lecon,Cycle de vie de la donnée et rôle du Data Anal...
5,DA-C1-L3-EXO,DA,C1 - Fondamentaux de la donnée,L1.3,Exercice - Cycle de vie de la donnée et rôle d...,exercice,Exercice - Cycle de vie de la donnée et rôle d...
6,DA-C1-QUIZ,DA,C1 - Fondamentaux de la donnée,Quiz C1,Quiz - Fondamentaux de la donnée,quiz,Quiz - Fondamentaux de la donnée C1 - Fondam...
7,DA-C2-L1-LEC,DA,C2 - Excel pour l'analyse,L2.1,Formules et fonctions essentielles,lecon,Formules et fonctions essentielles C2 - Exce...
8,DA-C2-L1-EXO,DA,C2 - Excel pour l'analyse,L2.1,Exercice - Formules et fonctions essentielles,exercice,Exercice - Formules et fonctions essentielles ...
9,DA-C2-L2-LEC,DA,C2 - Excel pour l'analyse,L2.2,Tableaux croisés dynamiques,lecon,Tableaux croisés dynamiques C2 - Excel pour ...


In [10]:
# ============================================================
# CELLULE 10 — Index TF-IDF
# ============================================================

VECTORISEUR = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1
)


MATRICE_RAG = (
    VECTORISEUR
    .fit_transform(
        RESSOURCES[
            "TEXTE_RAG"
        ]
    )
)


print(
    "Nombre de ressources :",
    MATRICE_RAG.shape[0]
)

print(
    "Nombre de termes :",
    MATRICE_RAG.shape[1]
)

Nombre de ressources : 168
Nombre de termes : 773


In [11]:
# ============================================================
# CELLULE 11 — Recherche des ressources existantes
# ============================================================

COLONNES_RESULTAT_RAG = [
    "RESOURCE_ID",
    "PARCOURS",
    "CHAPITRE",
    "LECON",
    "TITRE",
    "TYPE_RESSOURCE",
    "SOURCE",
    "SCORE_SIMILARITE"
]


def rechercher_ressources(
    parcours,
    requete,
    top_k=5
):

    if pd.isna(requete):

        return pd.DataFrame(
            columns=COLONNES_RESULTAT_RAG
        )


    requete = str(
        requete
    ).strip()


    parcours = str(
        parcours
    ).strip().upper()


    if requete == "":

        return pd.DataFrame(
            columns=COLONNES_RESULTAT_RAG
        )


    if top_k < 1:

        raise ValueError(
            "top_k doit être >= 1."
        )


    masque = (
        RESSOURCES[
            "PARCOURS"
        ]
        ==
        parcours
    )


    indices_parcours = np.where(
        masque.to_numpy()
    )[0]


    if len(indices_parcours) == 0:

        return pd.DataFrame(
            columns=COLONNES_RESULTAT_RAG
        )


    vecteur_requete = (
        VECTORISEUR
        .transform(
            [requete]
        )
    )


    scores = (
        cosine_similarity(
            vecteur_requete,
            MATRICE_RAG[
                indices_parcours
            ]
        )
        .ravel()
    )


    positions_valides = np.where(
        scores > 0
    )[0]


    if len(positions_valides) == 0:

        return pd.DataFrame(
            columns=COLONNES_RESULTAT_RAG
        )


    scores_valides = scores[
        positions_valides
    ]


    ordre_local = (
        np.argsort(
            scores_valides
        )[::-1]
        [:top_k]
    )


    positions_selectionnees = (
        positions_valides[
            ordre_local
        ]
    )


    indices_selectionnes = (
        indices_parcours[
            positions_selectionnees
        ]
    )


    scores_selectionnes = (
        scores[
            positions_selectionnees
        ]
    )


    resultats = (
        RESSOURCES
        .iloc[
            indices_selectionnes
        ]
        .copy()
    )


    resultats[
        "SCORE_SIMILARITE"
    ] = scores_selectionnes


    return (
        resultats[
            COLONNES_RESULTAT_RAG
        ]
        .reset_index(
            drop=True
        )
    )

In [12]:
# ============================================================
# CELLULE 12 — Premier test réel
# ============================================================

groupe_test = (
    RECOMMANDATIONS_GROUPES
    .iloc[0]
)


print(
    "Groupe :",
    groupe_test["ID_GROUPE"]
)

print(
    "Parcours :",
    groupe_test["PARCOURS"]
)

print(
    "Niveau :",
    groupe_test["NIVEAU"]
)

print(
    "Lacunes :",
    groupe_test["LACUNE_PRINCIPALE"]
)


resultats_test = (
    rechercher_ressources(
        parcours=
            groupe_test["PARCOURS"],

        requete=
            groupe_test["REQUETE_RAG"],

        top_k=5
    )
)


display(
    resultats_test
)

Groupe : G001
Parcours : BI
Niveau : Avance
Lacunes : Modèle en étoile | Relations Power BI | Cardinalité


,RESOURCE_ID,PARCOURS,CHAPITRE,LECON,TITRE,TYPE_RESSOURCE,SOURCE,SCORE_SIMILARITE
0,BI-C2-L2-LEC,BI,C2 - Modélisation dimensionnelle,L2.2,Schéma en étoile et en flocon,lecon,catalogue_titres_v1,0.223559
1,BI-C2-L2-EXO,BI,C2 - Modélisation dimensionnelle,L2.2,Exercice - Schéma en étoile et en flocon,exercice,catalogue_titres_v1,0.213532
2,BI-C5-QUIZ,BI,C5 - Outils de reporting (Power BI / Tableau),Quiz C5,Quiz - Outils de reporting (Power BI / Tableau),quiz,catalogue_titres_v1,0.176750
3,BI-C5-L1-LEC,BI,C5 - Outils de reporting (Power BI / Tableau),L5.1,Connexion et modélisation des données,lecon,catalogue_titres_v1,0.163420
4,BI-C5-L1-EXO,BI,C5 - Outils de reporting (Power BI / Tableau),L5.1,Exercice - Connexion et modélisation des données,exercice,catalogue_titres_v1,0.155770


In [13]:
# ============================================================
# CELLULE 13 — Requête retrieval ciblée
# ============================================================

def construire_requete_retrieval(row):

    lacune = row[
        "LACUNE_PRINCIPALE"
    ]


    if pd.isna(lacune):
        return pd.NA


    lacune = str(
        lacune
    ).strip()


    valeurs_sans_lacune = {
        "",
        "nan",
        "Aucune",
        "Aucune_lacune_T0",
        "Aucune_lacune_T1",
        "Aucune lacune T0 détectée",
        "Aucune lacune T1 détectée"
    }


    if lacune in valeurs_sans_lacune:
        return pd.NA


    return lacune


RECOMMANDATIONS_GROUPES[
    "REQUETE_RETRIEVAL"
] = (
    RECOMMANDATIONS_GROUPES
    .apply(
        construire_requete_retrieval,
        axis=1
    )
)


display(
    RECOMMANDATIONS_GROUPES[
        [
            "ID_GROUPE",
            "PARCOURS",
            "NIVEAU",
            "LACUNE_PRINCIPALE",
            "REQUETE_RETRIEVAL"
        ]
    ]
)

,ID_GROUPE,PARCOURS,NIVEAU,LACUNE_PRINCIPALE,REQUETE_RETRIEVAL
0,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,Modèle en étoile | Relations Power BI | Cardin...
1,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,Typage des clés | Nettoyage de cellules fusion...
2,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,Typage des clés | Relations plusieurs-à-plusie...
3,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Power Query (évaluation paresseuse) | Power Pi...
4,G005,DA,Debutant,Power Query (fusion),Power Query (fusion)
5,G006,DA,Intermediaire,Power Pivot (schéma en étoile) | Power Query (...,Power Pivot (schéma en étoile) | Power Query (...
6,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Machine Learning (déséquilibre) | Pandas (join...
7,G008,DS,Debutant,Machine Learning (généralisation) | Pandas (jo...,Machine Learning (généralisation) | Pandas (jo...
8,G009,DS,Intermediaire,Machine Learning (déséquilibre) | Statistiques...,Machine Learning (déséquilibre) | Statistiques...
9,G010,IA,Avance,Open WebUI (Skills) | LLM (paramètres) | Open ...,Open WebUI (Skills) | LLM (paramètres) | Open ...


Cellule 14 — Retester G001

In [14]:
# ============================================================
# CELLULE 14 — Test ciblé
# ============================================================

groupe_test = (
    RECOMMANDATIONS_GROUPES
    .loc[
        RECOMMANDATIONS_GROUPES[
            "ID_GROUPE"
        ] == "G001"
    ]
    .iloc[0]
)


resultats_test_cibles = (
    rechercher_ressources(
        parcours=
            groupe_test["PARCOURS"],

        requete=
            groupe_test["REQUETE_RETRIEVAL"],

        top_k=5
    )
)


print(
    "Groupe :",
    groupe_test["ID_GROUPE"]
)

print(
    "Requête :",
    groupe_test["REQUETE_RETRIEVAL"]
)


display(
    resultats_test_cibles
)   

Groupe : G001
Requête : Modèle en étoile | Relations Power BI | Cardinalité


,RESOURCE_ID,PARCOURS,CHAPITRE,LECON,TITRE,TYPE_RESSOURCE,SOURCE,SCORE_SIMILARITE
0,BI-C2-L2-LEC,BI,C2 - Modélisation dimensionnelle,L2.2,Schéma en étoile et en flocon,lecon,catalogue_titres_v1,0.359812
1,BI-C2-L2-EXO,BI,C2 - Modélisation dimensionnelle,L2.2,Exercice - Schéma en étoile et en flocon,exercice,catalogue_titres_v1,0.343674
2,BI-C5-QUIZ,BI,C5 - Outils de reporting (Power BI / Tableau),Quiz C5,Quiz - Outils de reporting (Power BI / Tableau),quiz,catalogue_titres_v1,0.249656
3,BI-C5-L3-LEC,BI,C5 - Outils de reporting (Power BI / Tableau),L5.3,Rapports interactifs,lecon,catalogue_titres_v1,0.207699
4,BI-C5-L3-EXO,BI,C5 - Outils de reporting (Power BI / Tableau),L5.3,Exercice - Rapports interactifs,exercice,catalogue_titres_v1,0.195985


In [15]:
# ============================================================
# CELLULE 15 — Retrieval de tous les groupes
# ============================================================

resultats_retrieval = []


for _, groupe in (
    RECOMMANDATIONS_GROUPES
    .iterrows()
):

    id_groupe = groupe[
        "ID_GROUPE"
    ]

    parcours = groupe[
        "PARCOURS"
    ]

    niveau = groupe[
        "NIVEAU"
    ]

    lacune = groupe[
        "LACUNE_PRINCIPALE"
    ]

    requete = groupe[
        "REQUETE_RETRIEVAL"
    ]


    # --------------------------------------------------------
    # Aucun besoin ciblé
    # --------------------------------------------------------

    if pd.isna(requete):

        resultats_retrieval.append(
            {
                "ID_GROUPE":
                    id_groupe,

                "PARCOURS":
                    parcours,

                "NIVEAU":
                    niveau,

                "LACUNE_PRINCIPALE":
                    lacune,

                "REQUETE_RETRIEVAL":
                    pd.NA,

                "RANG":
                    pd.NA,

                "RESOURCE_ID":
                    pd.NA,

                "CHAPITRE":
                    pd.NA,

                "LECON":
                    pd.NA,

                "TITRE":
                    pd.NA,

                "TYPE_RESSOURCE":
                    pd.NA,

                "SOURCE":
                    pd.NA,

                "SCORE_SIMILARITE":
                    pd.NA,

                "STATUT_RETRIEVAL":
                    "PAS_DE_LACUNE_CIBLEE"
            }
        )

        continue


    resultats = rechercher_ressources(
        parcours=parcours,
        requete=requete,
        top_k=5
    )


    # --------------------------------------------------------
    # Aucun chevauchement lexical
    # --------------------------------------------------------

    if resultats.empty:

        resultats_retrieval.append(
            {
                "ID_GROUPE":
                    id_groupe,

                "PARCOURS":
                    parcours,

                "NIVEAU":
                    niveau,

                "LACUNE_PRINCIPALE":
                    lacune,

                "REQUETE_RETRIEVAL":
                    requete,

                "RANG":
                    pd.NA,

                "RESOURCE_ID":
                    pd.NA,

                "CHAPITRE":
                    pd.NA,

                "LECON":
                    pd.NA,

                "TITRE":
                    pd.NA,

                "TYPE_RESSOURCE":
                    pd.NA,

                "SOURCE":
                    pd.NA,

                "SCORE_SIMILARITE":
                    0.0,

                "STATUT_RETRIEVAL":
                    "AUCUNE_CORRESPONDANCE_LEXICALE"
            }
        )

        continue


    # --------------------------------------------------------
    # Candidats
    # --------------------------------------------------------

    for rang, (_, ressource) in enumerate(
        resultats.iterrows(),
        start=1
    ):

        resultats_retrieval.append(
            {
                "ID_GROUPE":
                    id_groupe,

                "PARCOURS":
                    parcours,

                "NIVEAU":
                    niveau,

                "LACUNE_PRINCIPALE":
                    lacune,

                "REQUETE_RETRIEVAL":
                    requete,

                "RANG":
                    rang,

                "RESOURCE_ID":
                    ressource["RESOURCE_ID"],

                "CHAPITRE":
                    ressource["CHAPITRE"],

                "LECON":
                    ressource["LECON"],

                "TITRE":
                    ressource["TITRE"],

                "TYPE_RESSOURCE":
                    ressource[
                        "TYPE_RESSOURCE"
                    ],

                "SOURCE":
                    ressource["SOURCE"],

                "SCORE_SIMILARITE":
                    float(
                        ressource[
                            "SCORE_SIMILARITE"
                        ]
                    ),

                "STATUT_RETRIEVAL":
                    "CANDIDAT_RAG"
            }
        )


RETRIEVAL_GROUPES = pd.DataFrame(
    resultats_retrieval
)


candidats = (
    RETRIEVAL_GROUPES[
        RETRIEVAL_GROUPES[
            "STATUT_RETRIEVAL"
        ] == "CANDIDAT_RAG"
    ]
)


assert (
    candidats[
        "SCORE_SIMILARITE"
    ] > 0
).all()


assert (
    RETRIEVAL_GROUPES[
        "ID_GROUPE"
    ]
    .nunique()
    ==
    RECOMMANDATIONS_GROUPES[
        "ID_GROUPE"
    ]
    .nunique()
)


print(
    "Lignes retrieval :",
    len(RETRIEVAL_GROUPES)
)

print(
    "Groupes :",
    RETRIEVAL_GROUPES[
        "ID_GROUPE"
    ].nunique()
)

print(
    "Groupes avec candidat :",
    candidats[
        "ID_GROUPE"
    ].nunique()
)


display(
    RETRIEVAL_GROUPES[
        "STATUT_RETRIEVAL"
    ]
    .value_counts()
    .rename("N")
    .to_frame()
)

Lignes retrieval : 47
Groupes : 12
Groupes avec candidat : 11


,N
STATUT_RETRIEVAL,
CANDIDAT_RAG,46
AUCUNE_CORRESPONDANCE_LEXICALE,1


In [16]:
# ============================================================
# CELLULE 16 — Top 3 par groupe
# ============================================================

rang_numerique = pd.to_numeric(
    RETRIEVAL_GROUPES[
        "RANG"
    ],
    errors="coerce"
)


TOP3_GROUPES = (
    RETRIEVAL_GROUPES[
        rang_numerique.isna()
        |
        rang_numerique.le(3)
    ]
    .copy()
)


display(
    TOP3_GROUPES[
        [
            "ID_GROUPE",
            "PARCOURS",
            "NIVEAU",
            "LACUNE_PRINCIPALE",
            "RANG",
            "RESOURCE_ID",
            "TITRE",
            "TYPE_RESSOURCE",
            "SCORE_SIMILARITE"
        ]
    ]
)

,ID_GROUPE,PARCOURS,NIVEAU,LACUNE_PRINCIPALE,RANG,RESOURCE_ID,TITRE,TYPE_RESSOURCE,SCORE_SIMILARITE
0,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,1,BI-C2-L2-LEC,Schéma en étoile et en flocon,lecon,0.359812
1,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,2,BI-C2-L2-EXO,Exercice - Schéma en étoile et en flocon,exercice,0.343674
2,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,3,BI-C5-QUIZ,Quiz - Outils de reporting (Power BI / Tableau),quiz,0.249656
5,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,1,BI-C3-L3-LEC,Qualité des données,lecon,0.106246
6,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,2,BI-C3-L3-EXO,Exercice - Qualité des données,exercice,0.099577
7,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,3,BI-C6-L3-LEC,Déploiement et adoption des rapports,lecon,0.092163
10,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,1,BI-C5-QUIZ,Quiz - Outils de reporting (Power BI / Tableau),quiz,0.412176
11,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,2,BI-C5-L1-LEC,Connexion et modélisation des données,lecon,0.357060
12,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,3,BI-C5-L3-LEC,Rapports interactifs,lecon,0.342907
15,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,<NA>,<NA>,<NA>,<NA>,0.000000


In [17]:
# ============================================================
# CELLULE 17 — Audit du retrieval
# ============================================================

scores_valides = (
    RETRIEVAL_GROUPES[
        RETRIEVAL_GROUPES[
            "STATUT_RETRIEVAL"
        ] == "CANDIDAT_RAG"
    ]
    .copy()
)


resume_scores = (
    scores_valides
    .groupby(
        "RANG"
    )[
        "SCORE_SIMILARITE"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max"
        ]
    )
    .reset_index()
)


display(
    resume_scores.round(4)
)

,RANG,count,mean,median,min,max
0,1,11,0.2383,0.2130,0.0674,0.4122
1,2,11,0.2222,0.2015,0.0644,0.3794
2,3,8,0.1791,0.1683,0.0589,0.3429
3,4,8,0.1594,0.1491,0.0564,0.3403
4,5,8,0.1496,0.1418,0.0561,0.3236


Phase D — Human-in-the-loop
Cellule 18 — Statut des candidats

In [18]:
# ============================================================
# CELLULE 18 — Statut des candidats
# ============================================================

def statut_candidat(row):

    statut = str(
        row[
            "STATUT_RETRIEVAL"
        ]
    ).strip()


    if statut == "PAS_DE_LACUNE_CIBLEE":

        return (
            "PAS_DE_RECHERCHE_NECESSAIRE"
        )


    if statut == "AUCUNE_CORRESPONDANCE_LEXICALE":

        return (
            "AUCUNE_CORRESPONDANCE_LEXICALE"
        )


    if statut == "CANDIDAT_RAG":

        score = row[
            "SCORE_SIMILARITE"
        ]

        if pd.isna(score):

            return "CANDIDAT_INVALIDE"

        if float(score) <= 0:

            return (
                "AUCUNE_CORRESPONDANCE_LEXICALE"
            )

        return (
            "A_VALIDER_ENSEIGNANT"
        )


    return "STATUT_A_VERIFIER"


RETRIEVAL_GROUPES[
    "STATUT_CANDIDAT"
] = (
    RETRIEVAL_GROUPES
    .apply(
        statut_candidat,
        axis=1
    )
)


display(
    RETRIEVAL_GROUPES[
        "STATUT_CANDIDAT"
    ]
    .value_counts()
    .rename("N")
    .to_frame()
)

,N
STATUT_CANDIDAT,
A_VALIDER_ENSEIGNANT,46
AUCUNE_CORRESPONDANCE_LEXICALE,1


Cellule 19 — Construire le fichier de validation

In [19]:
# ============================================================
# CELLULE 19 — Table HITL
# ============================================================

rang_numerique = pd.to_numeric(
    RETRIEVAL_GROUPES[
        "RANG"
    ],
    errors="coerce"
)


HITL_RESSOURCES = (
    RETRIEVAL_GROUPES[
        rang_numerique.isna()
        |
        rang_numerique.le(3)
    ]
    .copy()
)


for col in [
    "DECISION_ENSEIGNANT",
    "COMMENTAIRE_ENSEIGNANT",
    "RESOURCE_ID_REMPLACEMENT",
    "TITRE_REMPLACEMENT"
]:

    HITL_RESSOURCES[
        col
    ] = ""


print(
    "Lignes HITL :",
    len(HITL_RESSOURCES)
)

print(
    "Groupes :",
    HITL_RESSOURCES[
        "ID_GROUPE"
    ].nunique()
)


display(
    HITL_RESSOURCES.head(20)
)

Lignes HITL : 31
Groupes : 12


,ID_GROUPE,PARCOURS,NIVEAU,LACUNE_PRINCIPALE,REQUETE_RETRIEVAL,RANG,RESOURCE_ID,CHAPITRE,LECON,TITRE,TYPE_RESSOURCE,SOURCE,SCORE_SIMILARITE,STATUT_RETRIEVAL,STATUT_CANDIDAT,DECISION_ENSEIGNANT,COMMENTAIRE_ENSEIGNANT,RESOURCE_ID_REMPLACEMENT,TITRE_REMPLACEMENT
0,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,Modèle en étoile | Relations Power BI | Cardin...,1,BI-C2-L2-LEC,C2 - Modélisation dimensionnelle,L2.2,Schéma en étoile et en flocon,lecon,catalogue_titres_v1,0.359812,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
1,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,Modèle en étoile | Relations Power BI | Cardin...,2,BI-C2-L2-EXO,C2 - Modélisation dimensionnelle,L2.2,Exercice - Schéma en étoile et en flocon,exercice,catalogue_titres_v1,0.343674,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
2,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,Modèle en étoile | Relations Power BI | Cardin...,3,BI-C5-QUIZ,C5 - Outils de reporting (Power BI / Tableau),Quiz C5,Quiz - Outils de reporting (Power BI / Tableau),quiz,catalogue_titres_v1,0.249656,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
5,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,Typage des clés | Nettoyage de cellules fusion...,1,BI-C3-L3-LEC,C3 - ETL et entrepôts de données,L3.3,Qualité des données,lecon,catalogue_titres_v1,0.106246,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
6,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,Typage des clés | Nettoyage de cellules fusion...,2,BI-C3-L3-EXO,C3 - ETL et entrepôts de données,L3.3,Exercice - Qualité des données,exercice,catalogue_titres_v1,0.099577,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
7,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,Typage des clés | Nettoyage de cellules fusion...,3,BI-C6-L3-LEC,C6 - Gouvernance et pilotage de la performance,L6.3,Déploiement et adoption des rapports,lecon,catalogue_titres_v1,0.092163,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
10,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,Typage des clés | Relations plusieurs-à-plusie...,1,BI-C5-QUIZ,C5 - Outils de reporting (Power BI / Tableau),Quiz C5,Quiz - Outils de reporting (Power BI / Tableau),quiz,catalogue_titres_v1,0.412176,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
11,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,Typage des clés | Relations plusieurs-à-plusie...,2,BI-C5-L1-LEC,C5 - Outils de reporting (Power BI / Tableau),L5.1,Connexion et modélisation des données,lecon,catalogue_titres_v1,0.357060,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
12,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,Typage des clés | Relations plusieurs-à-plusie...,3,BI-C5-L3-LEC,C5 - Outils de reporting (Power BI / Tableau),L5.3,Rapports interactifs,lecon,catalogue_titres_v1,0.342907,CANDIDAT_RAG,A_VALIDER_ENSEIGNANT,,,,
15,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Power Query (évaluation paresseuse) | Power Pi...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.000000,AUCUNE_CORRESPONDANCE_LEXICALE,AUCUNE_CORRESPONDANCE_LEXICALE,,,,


Cellule 20 — Export enseignant
Cette version protège ton fichier déjà rempli contre un écrasement accidentel.

In [20]:
# ============================================================
# CELLULE 20 — Export HITL RAG protégé
# ============================================================

LLM_DIR = OUTPUT_DIR / "llm"
KNOWLEDGE_STAGING_DIR = OUTPUT_DIR / "knowledge_staging"

for dossier in [
    LLM_DIR,
    KNOWLEDGE_STAGING_DIR
]:
    dossier.mkdir(
        parents=True,
        exist_ok=True
    )


FICHIER_HITL_RAG = (
    HITL_DIR
    / "validation_enseignant_ressources.csv"
)


def signature_hitl_rag(df):

    temp = df.copy()

    for col in [
        "ID_GROUPE",
        "RESOURCE_ID"
    ]:

        if col not in temp.columns:
            temp[col] = ""

        temp[col] = (
            temp[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )


    temp["RANG"] = pd.to_numeric(
        temp["RANG"],
        errors="coerce"
    ).fillna(-1).astype(int)


    return (
        temp[
            [
                "ID_GROUPE",
                "RANG",
                "RESOURCE_ID"
            ]
        ]
        .sort_values(
            [
                "ID_GROUPE",
                "RANG",
                "RESOURCE_ID"
            ]
        )
        .reset_index(drop=True)
    )


if FICHIER_HITL_RAG.exists():

    existant = pd.read_csv(
        FICHIER_HITL_RAG
    )

    if (
        "DECISION_ENSEIGNANT"
        not in existant.columns
    ):
        existant[
            "DECISION_ENSEIGNANT"
        ] = ""


    contient_decisions = (
        existant[
            "DECISION_ENSEIGNANT"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .any()
    )


    meme_retrieval = (
        signature_hitl_rag(
            existant
        )
        .equals(
            signature_hitl_rag(
                HITL_RESSOURCES
            )
        )
    )


    if (
        contient_decisions
        and
        meme_retrieval
    ):

        print(
            "Fichier RAG déjà validé : "
            "aucun écrasement."
        )

    elif (
        contient_decisions
        and
        not meme_retrieval
    ):

        raise RuntimeError(
            "Le fichier de validation RAG "
            "contient des décisions, mais "
            "le Top-3 actuel a changé."
        )

    else:

        HITL_RESSOURCES.to_csv(
            FICHIER_HITL_RAG,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "Fichier HITL RAG mis à jour."
        )


else:

    HITL_RESSOURCES.to_csv(
        FICHIER_HITL_RAG,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Fichier HITL RAG créé."
    )


print(
    FICHIER_HITL_RAG
)

Fichier RAG déjà validé : aucun écrasement.
/home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/06_RAG_LLM_HITL/hitl/validation_enseignant_ressources.csv


In [21]:
# ============================================================
# CELLULE 21 — Rechargement validation RAG
# ============================================================

if not FICHIER_HITL_RAG.exists():

    raise FileNotFoundError(
        FICHIER_HITL_RAG
    )


HITL_RESSOURCES = pd.read_csv(
    FICHIER_HITL_RAG
)


COLONNES_HITL_RAG = [
    "DECISION_ENSEIGNANT",
    "COMMENTAIRE_ENSEIGNANT",
    "RESOURCE_ID_REMPLACEMENT",
    "TITRE_REMPLACEMENT"
]


for col in COLONNES_HITL_RAG:

    if col not in HITL_RESSOURCES.columns:
        HITL_RESSOURCES[col] = ""

    HITL_RESSOURCES[col] = (
        HITL_RESSOURCES[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


HITL_RESSOURCES[
    "DECISION_ENSEIGNANT"
] = (
    HITL_RESSOURCES[
        "DECISION_ENSEIGNANT"
    ]
    .str.upper()
)


DECISIONS_RAG_AUTORISEES = {
    "",
    "VALIDE",
    "REJETE",
    "REMPLACE",
    "AUCUNE_RESSOURCE_ADAPTEE"
}


inconnues = (
    set(
        HITL_RESSOURCES[
            "DECISION_ENSEIGNANT"
        ].unique()
    )
    -
    DECISIONS_RAG_AUTORISEES
)


assert not inconnues, (
    f"Décisions RAG inconnues : {inconnues}"
)


# ------------------------------------------------------------
# Vérification des remplacements
# ------------------------------------------------------------

for _, ligne in (
    HITL_RESSOURCES[
        HITL_RESSOURCES[
            "DECISION_ENSEIGNANT"
        ] == "REMPLACE"
    ]
    .iterrows()
):

    replacement_id = (
        ligne[
            "RESOURCE_ID_REMPLACEMENT"
        ]
    )

    assert replacement_id != "", (
        "REMPLACE nécessite "
        "RESOURCE_ID_REMPLACEMENT."
    )


    candidat = (
        RESSOURCES[
            RESSOURCES[
                "RESOURCE_ID"
            ].astype(str)
            ==
            replacement_id
        ]
    )


    assert len(candidat) == 1, (
        "RESOURCE_ID_REMPLACEMENT "
        f"inconnu : {replacement_id}"
    )


    assert (
        str(
            candidat.iloc[0][
                "PARCOURS"
            ]
        ).strip()
        ==
        str(
            ligne[
                "PARCOURS"
            ]
        ).strip()
    ), (
        "La ressource de remplacement "
        "doit appartenir au même parcours."
    )


print(
    "Décisions RAG :"
)

display(
    HITL_RESSOURCES[
        "DECISION_ENSEIGNANT"
    ]
    .value_counts(
        dropna=False
    )
    .rename("N")
    .to_frame()
)

Décisions RAG :


,N
DECISION_ENSEIGNANT,
VALIDE,18
REJETE,11
REMPLACE,1
AUCUNE_RESSOURCE_ADAPTEE,1


Cellule 22 — État HITL des groupes

In [22]:
# ============================================================
# CELLULE 22 — Ressources existantes validées
# ============================================================

lignes_rag_validees = []


for _, ligne in (
    HITL_RESSOURCES
    .iterrows()
):

    decision = (
        str(
            ligne[
                "DECISION_ENSEIGNANT"
            ]
        )
        .strip()
        .upper()
    )


    # --------------------------------------------------------
    # Ressource RAG directement validée
    # --------------------------------------------------------

    if decision == "VALIDE":

        resource_id = (
            str(
                ligne[
                    "RESOURCE_ID"
                ]
            )
            .strip()
        )


        if (
            resource_id == ""
            or
            resource_id.lower() == "nan"
        ):
            continue


        lignes_rag_validees.append(
            {
                "ID_GROUPE":
                    ligne[
                        "ID_GROUPE"
                    ],

                "PARCOURS":
                    ligne[
                        "PARCOURS"
                    ],

                "RESOURCE_ID":
                    resource_id,

                "TITRE":
                    ligne[
                        "TITRE"
                    ],

                "TYPE_RESSOURCE":
                    ligne[
                        "TYPE_RESSOURCE"
                    ],

                "CHAPITRE":
                    ligne[
                        "CHAPITRE"
                    ],

                "LECON":
                    ligne[
                        "LECON"
                    ],

                "ORIGINE":
                    "KNOWLEDGE_RAG",

                "SOURCE_DECISION":
                    "VALIDE"
            }
        )


    # --------------------------------------------------------
    # Ressource existante choisie manuellement
    # --------------------------------------------------------

    elif decision == "REMPLACE":

        replacement_id = (
            str(
                ligne[
                    "RESOURCE_ID_REMPLACEMENT"
                ]
            )
            .strip()
        )


        ressource = (
            RESSOURCES[
                RESSOURCES[
                    "RESOURCE_ID"
                ].astype(str)
                ==
                replacement_id
            ]
            .iloc[0]
        )


        lignes_rag_validees.append(
            {
                "ID_GROUPE":
                    ligne[
                        "ID_GROUPE"
                    ],

                "PARCOURS":
                    ressource[
                        "PARCOURS"
                    ],

                "RESOURCE_ID":
                    ressource[
                        "RESOURCE_ID"
                    ],

                "TITRE":
                    ressource[
                        "TITRE"
                    ],

                "TYPE_RESSOURCE":
                    ressource[
                        "TYPE_RESSOURCE"
                    ],

                "CHAPITRE":
                    ressource[
                        "CHAPITRE"
                    ],

                "LECON":
                    ressource[
                        "LECON"
                    ],

                "ORIGINE":
                    "KNOWLEDGE_REMPLACEMENT",

                "SOURCE_DECISION":
                    "REMPLACE"
            }
        )


RESSOURCES_RAG_VALIDEES = (
    pd.DataFrame(
        lignes_rag_validees,
        columns=[
            "ID_GROUPE",
            "PARCOURS",
            "RESOURCE_ID",
            "TITRE",
            "TYPE_RESSOURCE",
            "CHAPITRE",
            "LECON",
            "ORIGINE",
            "SOURCE_DECISION"
        ]
    )
)


if not RESSOURCES_RAG_VALIDEES.empty:

    RESSOURCES_RAG_VALIDEES = (
        RESSOURCES_RAG_VALIDEES
        .drop_duplicates(
            subset=[
                "ID_GROUPE",
                "RESOURCE_ID"
            ]
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Statut par groupe
# ------------------------------------------------------------

statuts = []


for id_groupe, temp in (
    HITL_RESSOURCES
    .groupby(
        "ID_GROUPE",
        sort=False
    )
):

    decisions = (
        temp[
            "DECISION_ENSEIGNANT"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    if decisions.eq("").any():

        statut = (
            "EN_ATTENTE_VALIDATION_RAG"
        )

    else:

        n_validees = (
            RESSOURCES_RAG_VALIDEES[
                RESSOURCES_RAG_VALIDEES[
                    "ID_GROUPE"
                ]
                ==
                id_groupe
            ]
            .shape[0]
        )


        if n_validees > 0:

            statut = (
                "COUVERT_PAR_RAG"
            )

        else:

            statut = (
                "NON_COUVERT_PAR_RAG"
            )


    statuts.append(
        {
            "ID_GROUPE":
                id_groupe,

            "STATUT_RAG":
                statut
        }
    )


STATUT_RAG_GROUPES = (
    pd.DataFrame(
        statuts
    )
)


display(
    STATUT_RAG_GROUPES
)


print(
    "Ressources existantes validées :",
    len(
        RESSOURCES_RAG_VALIDEES
    )
)

,ID_GROUPE,STATUT_RAG
0,G001,COUVERT_PAR_RAG
1,G002,COUVERT_PAR_RAG
2,G003,COUVERT_PAR_RAG
3,G004,NON_COUVERT_PAR_RAG
4,G005,COUVERT_PAR_RAG
5,G006,COUVERT_PAR_RAG
6,G007,NON_COUVERT_PAR_RAG
7,G008,COUVERT_PAR_RAG
8,G009,COUVERT_PAR_RAG
9,G010,COUVERT_PAR_RAG


Ressources existantes validées : 19


Cellule 23 — Vérifier que HITL est terminé

In [23]:
# ============================================================
# CELLULE 23 — Besoins non couverts → fallback LLM
# ============================================================

VALIDATION_RAG_TERMINEE = (
    ~STATUT_RAG_GROUPES[
        "STATUT_RAG"
    ]
    .eq(
        "EN_ATTENTE_VALIDATION_RAG"
    )
    .any()
)


print(
    "Validation RAG terminée :",
    VALIDATION_RAG_TERMINEE
)


def colonne_ou_defaut(
    df,
    candidats,
    defaut=""
):

    for col in candidats:

        if col in df.columns:
            return df[col]

    return pd.Series(
        [defaut] * len(df),
        index=df.index
    )


GROUPES_CONTEXTES = (
    RECOMMANDATIONS_GROUPES
    .copy()
)


GROUPES_CONTEXTES[
    "PREREQUIS_LLM"
] = colonne_ou_defaut(
    GROUPES_CONTEXTES,
    [
        "PREREQUIS_LIMITANT",
        "PREREQUIS_DOMINANT"
    ],
    "Non renseigné"
)


GROUPES_CONTEXTES[
    "OBJECTIFS_LLM"
] = colonne_ou_defaut(
    GROUPES_CONTEXTES,
    [
        "OBJECTIFS_DOMINANTS"
    ],
    "Non renseigné"
)


if VALIDATION_RAG_TERMINEE:

    BESOINS_LLM = (
        STATUT_RAG_GROUPES[
            STATUT_RAG_GROUPES[
                "STATUT_RAG"
            ]
            ==
            "NON_COUVERT_PAR_RAG"
        ]
        .merge(
            GROUPES_CONTEXTES[
                [
                    "ID_GROUPE",
                    "PARCOURS",
                    "PROGRAMME_COMMUN",
                    "NIVEAU",
                    "LACUNE_PRINCIPALE",
                    "PREREQUIS_LLM",
                    "OBJECTIFS_LLM"
                ]
            ],
            on="ID_GROUPE",
            how="left",
            validate="one_to_one"
        )
    )


else:

    BESOINS_LLM = pd.DataFrame(
        columns=[
            "ID_GROUPE",
            "STATUT_RAG",
            "PARCOURS",
            "PROGRAMME_COMMUN",
            "NIVEAU",
            "LACUNE_PRINCIPALE",
            "PREREQUIS_LLM",
            "OBJECTIFS_LLM"
        ]
    )


print(
    "Groupes nécessitant le LLM :",
    len(
        BESOINS_LLM
    )
)


display(
    BESOINS_LLM
)

Validation RAG terminée : True


Groupes nécessitant le LLM : 2


,ID_GROUPE,STATUT_RAG,PARCOURS,PROGRAMME_COMMUN,NIVEAU,LACUNE_PRINCIPALE,PREREQUIS_LLM,OBJECTIFS_LLM
0,G004,NON_COUVERT_PAR_RAG,DA,PROGRAMME_DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,Entrepreneur | Developpeur Python / IA
1,G007,NON_COUVERT_PAR_RAG,DS,PROGRAMME_DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,Developpeur Python / IA | Data Analyst


Phase E — Évaluation du retrieval
Cellule 24 — Precision@3, Hit@3 et MRR@3

In [24]:
# ============================================================
# CELLULE 24 — Evaluation scientifique du retrieval RAG
# ============================================================

METRIQUES_RAG_GROUPES = pd.DataFrame()
METRIQUES_RAG_GLOBALES = pd.DataFrame()


if not VALIDATION_RAG_TERMINEE:

    print(
        "Métriques RAG non calculées : "
        "validation enseignant incomplète."
    )


else:

    candidats = (
        HITL_RESSOURCES[
            pd.to_numeric(
                HITL_RESSOURCES[
                    "RANG"
                ],
                errors="coerce"
            )
            .notna()
        ]
        .copy()
    )


    candidats[
        "RANG"
    ] = pd.to_numeric(
        candidats[
            "RANG"
        ],
        errors="coerce"
    )


    candidats = (
        candidats[
            candidats[
                "RANG"
            ] <= 3
        ]
        .copy()
    )


    # Une ressource initialement retrouvée
    # est pertinente seulement si l'enseignant
    # l'a directement validée.
    #
    # REMPLACE ne valide pas le résultat
    # initial du moteur RAG.

    candidats[
        "PERTINENT"
    ] = (
        candidats[
            "DECISION_ENSEIGNANT"
        ]
        ==
        "VALIDE"
    )


    groupes_recherche = (
        HITL_RESSOURCES[
            [
                "ID_GROUPE"
            ]
        ]
        .drop_duplicates()
    )


    lignes_metriques = []


    for id_groupe in (
        groupes_recherche[
            "ID_GROUPE"
        ]
    ):

        temp = (
            candidats[
                candidats[
                    "ID_GROUPE"
                ]
                ==
                id_groupe
            ]
            .sort_values(
                "RANG"
            )
        )


        n_pertinents = int(
            temp[
                "PERTINENT"
            ].sum()
        )


        precision_at_3 = (
            n_pertinents
            /
            3.0
        )


        hit_at_3 = int(
            n_pertinents > 0
        )


        if n_pertinents > 0:

            premier_rang = (
                temp.loc[
                    temp[
                        "PERTINENT"
                    ],
                    "RANG"
                ]
                .min()
            )

            mrr_at_3 = (
                1.0
                /
                float(
                    premier_rang
                )
            )

        else:

            mrr_at_3 = 0.0


        lignes_metriques.append(
            {
                "ID_GROUPE":
                    id_groupe,

                "N_CANDIDATS_RETOURNES":
                    len(temp),

                "N_PERTINENTS_TOP3":
                    n_pertinents,

                "PRECISION_AT_3":
                    precision_at_3,

                "HIT_AT_3":
                    hit_at_3,

                "MRR_AT_3":
                    mrr_at_3
            }
        )


    METRIQUES_RAG_GROUPES = (
        pd.DataFrame(
            lignes_metriques
        )
    )


    METRIQUES_RAG_GLOBALES = (
        pd.DataFrame(
            [
                {
                    "N_GROUPES":
                        len(
                            METRIQUES_RAG_GROUPES
                        ),

                    "PRECISION_AT_3":
                        METRIQUES_RAG_GROUPES[
                            "PRECISION_AT_3"
                        ].mean(),

                    "HIT_AT_3":
                        METRIQUES_RAG_GROUPES[
                            "HIT_AT_3"
                        ].mean(),

                    "MRR_AT_3":
                        METRIQUES_RAG_GROUPES[
                            "MRR_AT_3"
                        ].mean()
                }
            ]
        )
    )


    display(
        METRIQUES_RAG_GROUPES
    )

    display(
        METRIQUES_RAG_GLOBALES
    )

,ID_GROUPE,N_CANDIDATS_RETOURNES,N_PERTINENTS_TOP3,PRECISION_AT_3,HIT_AT_3,MRR_AT_3
0,G001,3,2,0.666667,1,1.000000
1,G002,3,2,0.666667,1,1.000000
2,G003,3,1,0.333333,1,0.500000
3,G004,0,0,0.000000,0,0.000000
4,G005,2,2,0.666667,1,1.000000
5,G006,2,2,0.666667,1,1.000000
6,G007,2,0,0.000000,0,0.000000
7,G008,3,2,0.666667,1,1.000000
8,G009,3,2,0.666667,1,1.000000
9,G010,3,1,0.333333,1,0.333333


,N_GROUPES,PRECISION_AT_3,HIT_AT_3,MRR_AT_3
0,12,0.5,0.833333,0.736111


# Phase LLM : uniquement pour les groupes non couverts


In [25]:
# ============================================================
# CELLULE 25A — Prompts LLM pour besoins non couverts
# ============================================================

def titres_knowledge_par_parcours(
    parcours
):

    temp = (
        RESSOURCES[
            RESSOURCES[
                "PARCOURS"
            ].astype(str)
            ==
            str(parcours)
        ]
    )


    return "\n".join(
        [
            f"- [{row.TYPE_RESSOURCE}] "
            f"{row.TITRE}"

            for row in (
                temp
                .itertuples()
            )
        ]
    )


def construire_prompt_generation(
    row
):

    titres_existants = (
        titres_knowledge_par_parcours(
            row[
                "PARCOURS"
            ]
        )
    )


    return f"""
Tu es un assistant pédagogique.

Le moteur RAG a recherché des ressources dans le Knowledge Base,
mais aucune ressource existante n'a été validée par l'enseignant
pour ce groupe.

Tu dois uniquement PROPOSER DE NOUVEAUX TITRES DE RESSOURCES.
Tu ne dois pas rédiger le contenu complet.

CONTEXTE DU GROUPE
------------------
ID groupe : {row['ID_GROUPE']}
Parcours : {row['PARCOURS']}
Programme : {row['PROGRAMME_COMMUN']}
Niveau exploratoire : {row['NIVEAU']}
Lacunes prioritaires : {row['LACUNE_PRINCIPALE']}
Prérequis : {row['PREREQUIS_LLM']}
Objectifs : {row['OBJECTIFS_LLM']}

RESSOURCES DEJA EXISTANTES DANS LE KNOWLEDGE
--------------------------------------------
{titres_existants}

CONTRAINTES
-----------
- rester strictement dans le parcours {row['PARCOURS']};
- ne pas créer un nouveau programme;
- ne pas changer les objectifs pédagogiques du parcours;
- cibler directement les lacunes indiquées;
- tenir compte du niveau et des prérequis;
- ne pas dupliquer un titre déjà présent dans le Knowledge Base;
- proposer exactement 3 ressources candidates :
  1 leçon,
  1 exercice,
  1 quiz;
- ne rédiger aucun contenu pédagogique complet;
- ces propositions ne sont PAS validées;
- elles devront obligatoirement être évaluées par un enseignant.

Retourner uniquement un JSON valide sous cette forme :

[
  {{
    "type_ressource": "lecon",
    "titre": "...",
    "lacune_couverte": "...",
    "justification": "..."
  }},
  {{
    "type_ressource": "exercice",
    "titre": "...",
    "lacune_couverte": "...",
    "justification": "..."
  }},
  {{
    "type_ressource": "quiz",
    "titre": "...",
    "lacune_couverte": "...",
    "justification": "..."
  }}
]
""".strip()


PROMPTS_GENERATION_LLM = (
    BESOINS_LLM
    .copy()
)


if not PROMPTS_GENERATION_LLM.empty:

    PROMPTS_GENERATION_LLM[
        "PROMPT_LLM"
    ] = (
        PROMPTS_GENERATION_LLM
        .apply(
            construire_prompt_generation,
            axis=1
        )
    )


FICHIER_PROMPTS_LLM = (
    LLM_DIR
    / "prompts_generation_titres.csv"
)


PROMPTS_GENERATION_LLM.to_csv(
    FICHIER_PROMPTS_LLM,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Prompts LLM :",
    len(
        PROMPTS_GENERATION_LLM
    )
)

print(
    FICHIER_PROMPTS_LLM
)

Prompts LLM : 2
/home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/06_RAG_LLM_HITL/llm/prompts_generation_titres.csv


In [26]:
# ============================================================
# CELLULE 25A-TEST — Vérifier Ollama et le modèle
# ============================================================

import requests

OLLAMA_URL = "http://localhost:11434"

# 1. Vérifier le serveur Ollama
r = requests.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10
)

print("GET /api/tags :", r.status_code)
print(r.text)

if r.status_code != 200:
    raise RuntimeError(
        "Le service présent sur localhost:11434 "
        "ne répond pas comme une API Ollama."
    )

# 2. Lister les modèles installés
data = r.json()

MODELES_OLLAMA = [
    m.get("name") or m.get("model")
    for m in data.get("models", [])
]

print("\nModèles Ollama disponibles :")
for m in MODELES_OLLAMA:
    print(" -", m)

if not MODELES_OLLAMA:
    raise RuntimeError(
        "Ollama fonctionne mais aucun modèle local n'est installé."
    )

GET /api/tags : 200
{"models":[{"name":"nomic-embed-text:latest","model":"nomic-embed-text:latest","modified_at":"2026-01-19T11:07:38.632452582+03:00","size":274302450,"digest":"0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f","details":{"parent_model":"","format":"gguf","family":"nomic-bert","families":["nomic-bert"],"parameter_size":"137M","quantization_level":"F16","context_length":2048,"embedding_length":768},"capabilities":["embedding"]},{"name":"mistral:latest","model":"mistral:latest","modified_at":"2026-01-15T16:35:58.912748649+03:00","size":4372824384,"digest":"6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"7.2B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":4096},"capabilities":["completion","tools"]}]}

Modèles Ollama disponibles :
 - nomic-embed-text:latest
 - mistral:latest


In [29]:
# ============================================================
# CELLULE 25A bis — Connexion Ollama définitive
# ============================================================

import requests


OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"


# ------------------------------------------------------------
# 1. Vérifier Ollama et récupérer les modèles installés
# ------------------------------------------------------------

r = requests.get(
    f"{OLLAMA_BASE_URL}/api/tags",
    timeout=10
)

if r.status_code != 200:

    raise RuntimeError(
        f"Ollama inaccessible : HTTP {r.status_code}\n"
        f"{r.text}"
    )


data_ollama = r.json()


MODELES_OLLAMA = [
    m.get("name") or m.get("model")
    for m in data_ollama.get("models", [])
]


if not MODELES_OLLAMA:

    raise RuntimeError(
        "Ollama fonctionne mais aucun modèle n'est installé."
    )


print(
    "Modèles Ollama disponibles :",
    MODELES_OLLAMA
)


# ------------------------------------------------------------
# 2. Sélection automatique d'un modèle génératif
# ------------------------------------------------------------

MODELES_PREFERES = [
    "qwen2.5:3b",
    "mistral:latest",
    "llama3.1:8b"
]


OLLAMA_MODEL = next(
    (
        modele
        for modele in MODELES_PREFERES
        if modele in MODELES_OLLAMA
    ),
    None
)


# Éviter un modèle embeddings uniquement
if OLLAMA_MODEL is None:

    candidats = [
        modele
        for modele in MODELES_OLLAMA
        if "embed" not in modele.lower()
    ]

    if not candidats:

        raise RuntimeError(
            "Aucun modèle génératif Ollama disponible."
        )

    OLLAMA_MODEL = candidats[0]


print(
    "Modèle LLM retenu :",
    OLLAMA_MODEL
)


# ------------------------------------------------------------
# 3. Générateur
# ------------------------------------------------------------

def GENERATEUR_LLM(prompt: str) -> str:

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,

        "options": {
            "temperature": 0.2,
            "num_predict": 700,
            "num_ctx": 4096
        },

        "keep_alive": "10m"
    }


    reponse = requests.post(
        OLLAMA_GENERATE_URL,
        json=payload,
        timeout=(10, 900)
    )


    if reponse.status_code != 200:

        raise RuntimeError(
            f"Ollama HTTP {reponse.status_code}\n"
            f"Modèle : {OLLAMA_MODEL}\n"
            f"Réponse : {reponse.text}"
        )


    data = reponse.json()

    texte = str(
        data.get(
            "response",
            ""
        )
    ).strip()


    if texte == "":

        raise RuntimeError(
            "Ollama a retourné une réponse vide."
        )


    return texte


print(
    "GENERATEUR_LLM connecté :",
    callable(GENERATEUR_LLM)
)

Modèles Ollama disponibles : ['nomic-embed-text:latest', 'mistral:latest']
Modèle LLM retenu : mistral:latest
GENERATEUR_LLM connecté : True


In [32]:
# ============================================================
# CELLULE 25B — Génération de nouveaux titres candidats
# uniquement pour les groupes NON couverts par le RAG
# (corrigée : retries, cache, sauvegarde progressive,
#  validation tolérante : garde 1 leçon + 1 exercice + 1 quiz)
# ============================================================

import json
import re
import time
import unicodedata


MAX_TENTATIVES = 3

SUFFIXE_PROMPT = (
    "\n\nIMPORTANT : réponds en français, en JSON compact. "
    "Chaque justification tient en UNE phrase courte (25 mots maximum). "
    "Aucun texte en dehors du JSON."
)

COLONNES_PROPOSITIONS_LLM = [
    "ID_GROUPE", "PARCOURS", "NIVEAU", "LACUNE_PRINCIPALE", "PREREQUIS",
    "CANDIDAT_LLM_ID", "TYPE_RESSOURCE_PROPOSE", "TITRE_PROPOSE",
    "LACUNE_COUVERTE", "JUSTIFICATION_LLM", "MODELE_LLM", "STATUT_CANDIDAT_LLM"
]

COLONNES_BRUTES_LLM = [
    "ID_GROUPE", "PARCOURS", "MODELE_LLM", "PROMPT_LLM",
    "REPONSE_LLM_BRUTE", "STATUT_GENERATION", "ERREUR_GENERATION"
]


# ------------------------------------------------------------
# Normalisations
# ------------------------------------------------------------

def normaliser_titre(valeur):

    if pd.isna(valeur):
        return ""

    texte = unicodedata.normalize("NFKD", str(valeur).strip().lower())
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    texte = re.sub(r"[^a-z0-9]+", " ", texte)

    return re.sub(r"\s+", " ", texte).strip()


def normaliser_type_ressource(valeur):
    """leçon / Leçon / lecon / Lesson -> lecon ; exercice ; quiz. Sinon ''."""

    t = normaliser_titre(valeur)

    if t.startswith("lec") or t.startswith("lesson") or t == "cours":
        return "lecon"

    if t.startswith("exer"):
        return "exercice"

    if t.startswith("quiz") or t == "qcm":
        return "quiz"

    return ""


TITRES_KNOWLEDGE_NORMALISES = set(
    RESSOURCES["TITRE"].dropna().map(normaliser_titre)
)


# ------------------------------------------------------------
# Extraction JSON
# ------------------------------------------------------------

def extraire_json_llm(texte):

    texte = str(texte).strip()
    texte = re.sub(r"^\s*```(?:json)?\s*", "", texte, flags=re.IGNORECASE)
    texte = re.sub(r"\s*```\s*$", "", texte)

    try:
        objet = json.loads(texte)

    except json.JSONDecodeError:

        debut, fin = texte.find("["), texte.rfind("]")

        if debut == -1 or fin <= debut:
            raise ValueError("Aucun tableau JSON trouvé (réponse tronquée ?).")

        objet = json.loads(texte[debut:fin + 1])

    if isinstance(objet, dict):

        listes = [v for v in objet.values() if isinstance(v, list)]

        if "propositions" in objet:
            objet = objet["propositions"]
        elif len(listes) == 1:
            objet = listes[0]
        else:
            raise ValueError("Objet JSON sans liste de propositions.")

    if not isinstance(objet, list):
        raise ValueError("La réponse LLM doit être une liste JSON.")

    return objet


# ------------------------------------------------------------
# Validation tolérante : garde la 1re leçon, le 1er exercice,
# le 1er quiz valides ; ignore le surplus et les éléments invalides.
# ------------------------------------------------------------

def valider_propositions_llm(propositions):

    if not isinstance(propositions, list):
        raise ValueError("La réponse LLM doit être une liste JSON.")

    retenues = {}

    for p in propositions:

        if not isinstance(p, dict):
            continue

        if any(
            c not in p
            for c in ["type_ressource", "titre", "lacune_couverte", "justification"]
        ):
            continue

        type_r = normaliser_type_ressource(p["type_ressource"])

        if type_r == "" or str(p["titre"]).strip() == "":
            continue

        if type_r not in retenues:
            p["type_ressource"] = type_r
            retenues[type_r] = p

    if set(retenues) != {"lecon", "exercice", "quiz"}:
        raise ValueError(
            f"Types obtenus {sorted(retenues)} sur {len(propositions)} éléments ; "
            "il faut leçon + exercice + quiz."
        )

    # exactement 3 propositions, dans un ordre fixe
    propositions[:] = [
        retenues["lecon"],
        retenues["exercice"],
        retenues["quiz"]
    ]


# ------------------------------------------------------------
# Cache : on ne régénère pas un groupe déjà OK (même prompt, même modèle)
# ------------------------------------------------------------

FICHIER_PROPOSITIONS_LLM = LLM_DIR / "propositions_titres_llm.csv"
FICHIER_SORTIES_LLM_BRUTES = LLM_DIR / "sorties_llm_brutes.csv"

CACHE_LLM = {}

if FICHIER_SORTIES_LLM_BRUTES.exists():
    try:
        ancien = pd.read_csv(FICHIER_SORTIES_LLM_BRUTES, dtype=str).fillna("")
        for _, a in ancien.iterrows():
            if a["STATUT_GENERATION"] == "OK" and a["MODELE_LLM"] == OLLAMA_MODEL:
                CACHE_LLM[(a["ID_GROUPE"], a["PROMPT_LLM"])] = a["REPONSE_LLM_BRUTE"]
    except Exception as e:
        print("Cache LLM ignoré :", type(e).__name__, e)


# ------------------------------------------------------------
# Génération
# ------------------------------------------------------------

lignes_propositions = []
lignes_brutes = []

print("Nombre de groupes à traiter par LLM :", len(PROMPTS_GENERATION_LLM))

for _, groupe in PROMPTS_GENERATION_LLM.iterrows():

    id_groupe = str(groupe["ID_GROUPE"])
    prompt = str(groupe["PROMPT_LLM"]) + SUFFIXE_PROMPT

    print("\nGénération :", id_groupe)

    propositions = None
    reponse = ""
    erreur = ""

    # 1) reprise depuis le cache
    cle = (id_groupe, prompt)

    if cle in CACHE_LLM:
        try:
            reponse = CACHE_LLM[cle]
            propositions = extraire_json_llm(reponse)
            valider_propositions_llm(propositions)
            print("  Repris du cache (aucun appel LLM).")
        except Exception:
            propositions = None

    # 2) sinon appel LLM avec nouvelles tentatives
    if propositions is None:

        for tentative in range(1, MAX_TENTATIVES + 1):

            t0 = time.time()

            try:
                reponse = GENERATEUR_LLM(prompt)
                propositions = extraire_json_llm(reponse)
                valider_propositions_llm(propositions)
                erreur = ""
                print(f"  OK (tentative {tentative}, {time.time() - t0:.0f} s)")
                break

            except Exception as e:
                propositions = None
                erreur = (
                    f"tentative {tentative}/{MAX_TENTATIVES} : "
                    f"{type(e).__name__}: {e}"
                )
                print("  ECHEC", erreur)
                print("  Réponse brute :", reponse[:300].replace("\n", " "))

    # 3) enregistrement
    lignes_brutes.append({
        "ID_GROUPE": id_groupe,
        "PARCOURS": groupe["PARCOURS"],
        "MODELE_LLM": OLLAMA_MODEL,
        "PROMPT_LLM": prompt,
        "REPONSE_LLM_BRUTE": reponse,
        "STATUT_GENERATION": "OK" if propositions is not None else "ERREUR",
        "ERREUR_GENERATION": erreur
    })

    if propositions is not None:

        titres_vus_groupe = set()

        for i, p in enumerate(propositions, start=1):

            titre = str(p["titre"]).strip()
            titre_norm = normaliser_titre(titre)

            if titre_norm in TITRES_KNOWLEDGE_NORMALISES:
                statut = "DOUBLON_KNOWLEDGE"
            elif titre_norm in titres_vus_groupe:
                statut = "DOUBLON_GENERATION"
            else:
                statut = "A_VALIDER_ENSEIGNANT"

            titres_vus_groupe.add(titre_norm)

            lignes_propositions.append({
                "ID_GROUPE": id_groupe,
                "PARCOURS": groupe["PARCOURS"],
                "NIVEAU": groupe["NIVEAU"],
                "LACUNE_PRINCIPALE": groupe["LACUNE_PRINCIPALE"],
                "PREREQUIS": groupe.get("PREREQUIS_LLM", ""),
                "CANDIDAT_LLM_ID": f"LLM-{id_groupe}-{i:02d}",
                "TYPE_RESSOURCE_PROPOSE": p["type_ressource"],
                "TITRE_PROPOSE": titre,
                "LACUNE_COUVERTE": str(p.get("lacune_couverte", "")).strip(),
                "JUSTIFICATION_LLM": str(p.get("justification", "")).strip(),
                "MODELE_LLM": OLLAMA_MODEL,
                "STATUT_CANDIDAT_LLM": statut
            })

    # sauvegarde progressive : rien n'est perdu si le notebook s'arrête
    pd.DataFrame(lignes_brutes, columns=COLONNES_BRUTES_LLM).to_csv(
        FICHIER_SORTIES_LLM_BRUTES, index=False, encoding="utf-8-sig"
    )


PROPOSITIONS_LLM = pd.DataFrame(lignes_propositions, columns=COLONNES_PROPOSITIONS_LLM)
SORTIES_LLM_BRUTES = pd.DataFrame(lignes_brutes, columns=COLONNES_BRUTES_LLM)

PROPOSITIONS_LLM_A_VALIDER = PROPOSITIONS_LLM[
    PROPOSITIONS_LLM["STATUT_CANDIDAT_LLM"] == "A_VALIDER_ENSEIGNANT"
].copy()

PROPOSITIONS_LLM.to_csv(FICHIER_PROPOSITIONS_LLM, index=False, encoding="utf-8-sig")


groupes_en_erreur = SORTIES_LLM_BRUTES.loc[
    SORTIES_LLM_BRUTES["STATUT_GENERATION"] == "ERREUR", "ID_GROUPE"
].tolist()

print("\nPropositions LLM totales :", len(PROPOSITIONS_LLM))
print("Propositions soumises à l'enseignant :", len(PROPOSITIONS_LLM_A_VALIDER))

if groupes_en_erreur:
    print(
        "\nGROUPES EN ERREUR :", groupes_en_erreur,
        "\n-> relancer cette cellule : les groupes déjà OK sont repris du cache."
    )

display(PROPOSITIONS_LLM)

Nombre de groupes à traiter par LLM : 2

Génération : G004
  OK (tentative 1, 247 s)

Génération : G007
  OK (tentative 1, 280 s)

Propositions LLM totales : 6
Propositions soumises à l'enseignant : 6


,ID_GROUPE,PARCOURS,NIVEAU,LACUNE_PRINCIPALE,PREREQUIS,CANDIDAT_LLM_ID,TYPE_RESSOURCE_PROPOSE,TITRE_PROPOSE,LACUNE_COUVERTE,JUSTIFICATION_LLM,MODELE_LLM,STATUT_CANDIDAT_LLM
0,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,LLM-G004-01,lecon,Power Query : Introduction et manipulation de ...,Power Query (évaluation paresseuse),Enseigner les bases de Power Query pour optimi...,mistral:latest,A_VALIDER_ENSEIGNANT
1,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,LLM-G004-02,exercice,Exercice : Power Query - Manipulation de données,Power Query (évaluation paresseuse),Exercice pratique pour appliquer les notions a...,mistral:latest,A_VALIDER_ENSEIGNANT
2,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,LLM-G004-03,quiz,Quiz : Power Query - Manipulation de données,Power Query (évaluation paresseuse),Quiz pour vérifier la compréhension des notion...,mistral:latest,A_VALIDER_ENSEIGNANT
3,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,LLM-G007-01,lecon,Machine Learning : Prétraitement des données,Machine Learning (découpage),Enseigner les techniques de prétraitement des ...,mistral:latest,A_VALIDER_ENSEIGNANT
4,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,LLM-G007-02,exercice,Exercice - Prétraitement des données,Machine Learning (découpage),Permettre aux étudiants de pratiquer les techn...,mistral:latest,A_VALIDER_ENSEIGNANT
5,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,LLM-G007-03,quiz,Quiz - Prétraitement des données,Machine Learning (découpage),Évaluer la compréhension des étudiants des tec...,mistral:latest,A_VALIDER_ENSEIGNANT


In [33]:
# ============================================================
# CELLULE 25C — Export HITL des propositions LLM
# ============================================================

FICHIER_VALIDATION_LLM = (
    LLM_DIR
    / "validation_enseignant_llm.csv"
)


COLONNES_VALIDATION_LLM = [

    "ID_GROUPE",
    "PARCOURS",
    "NIVEAU",

    "LACUNE_PRINCIPALE",
    "PREREQUIS",

    "CANDIDAT_LLM_ID",

    "TYPE_RESSOURCE_PROPOSE",
    "TITRE_PROPOSE",

    "LACUNE_COUVERTE",
    "JUSTIFICATION_LLM",

    "MODELE_LLM",

    "DECISION_ENSEIGNANT",
    "COMMENTAIRE_ENSEIGNANT",

    "TYPE_RESSOURCE_FINAL",
    "TITRE_FINAL"
]


VALIDATION_LLM = (
    PROPOSITIONS_LLM_A_VALIDER.copy()
)


for colonne in [

    "DECISION_ENSEIGNANT",
    "COMMENTAIRE_ENSEIGNANT",
    "TYPE_RESSOURCE_FINAL",
    "TITRE_FINAL"

]:

    if colonne not in VALIDATION_LLM.columns:

        VALIDATION_LLM[
            colonne
        ] = ""


VALIDATION_LLM = (
    VALIDATION_LLM[
        COLONNES_VALIDATION_LLM
    ]
)


# ------------------------------------------------------------
# Protection contre l'écrasement d'une validation existante
# ------------------------------------------------------------

if FICHIER_VALIDATION_LLM.exists():

    existant = pd.read_csv(
        FICHIER_VALIDATION_LLM
    )


    if (
        "DECISION_ENSEIGNANT"
        in existant.columns
        and
        existant[
            "DECISION_ENSEIGNANT"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .any()
    ):

        print(
            "Fichier de validation LLM déjà rempli : "
            "il n'est pas écrasé."
        )

        VALIDATION_LLM = existant


    else:

        VALIDATION_LLM.to_csv(
            FICHIER_VALIDATION_LLM,
            index=False,
            encoding="utf-8-sig"
        )

else:

    VALIDATION_LLM.to_csv(
        FICHIER_VALIDATION_LLM,
        index=False,
        encoding="utf-8-sig"
    )


print(
    "Propositions LLM à valider :",
    len(
        VALIDATION_LLM
    )
)

print(
    "Fichier enseignant :",
    FICHIER_VALIDATION_LLM
)

display(
    VALIDATION_LLM
)

Propositions LLM à valider : 6
Fichier enseignant : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/06_RAG_LLM_HITL/llm/validation_enseignant_llm.csv


,ID_GROUPE,PARCOURS,NIVEAU,LACUNE_PRINCIPALE,PREREQUIS,CANDIDAT_LLM_ID,TYPE_RESSOURCE_PROPOSE,TITRE_PROPOSE,LACUNE_COUVERTE,JUSTIFICATION_LLM,MODELE_LLM,DECISION_ENSEIGNANT,COMMENTAIRE_ENSEIGNANT,TYPE_RESSOURCE_FINAL,TITRE_FINAL
0,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,LLM-G004-01,lecon,Power Query : Introduction et manipulation de ...,Power Query (évaluation paresseuse),Enseigner les bases de Power Query pour optimi...,mistral:latest,,,,
1,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,LLM-G004-02,exercice,Exercice : Power Query - Manipulation de données,Power Query (évaluation paresseuse),Exercice pratique pour appliquer les notions a...,mistral:latest,,,,
2,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,LLM-G004-03,quiz,Quiz : Power Query - Manipulation de données,Power Query (évaluation paresseuse),Quiz pour vérifier la compréhension des notion...,mistral:latest,,,,
3,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,LLM-G007-01,lecon,Machine Learning : Prétraitement des données,Machine Learning (découpage),Enseigner les techniques de prétraitement des ...,mistral:latest,,,,
4,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,LLM-G007-02,exercice,Exercice - Prétraitement des données,Machine Learning (découpage),Permettre aux étudiants de pratiquer les techn...,mistral:latest,,,,
5,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,LLM-G007-03,quiz,Quiz - Prétraitement des données,Machine Learning (découpage),Évaluer la compréhension des étudiants des tec...,mistral:latest,,,,


Deuxième Human-in-the-loop

In [36]:
# ============================================================
# CELLULE 26 — Export HITL des candidats LLM
# (corrigée : fichier périmé archivé, lecture CSV robuste)
# ============================================================

from datetime import datetime

FICHIER_HITL_LLM = HITL_DIR / "validation_enseignant_llm.csv"

COLONNES_DECISION_LLM = [
    "DECISION_ENSEIGNANT_LLM", "TYPE_RESSOURCE_MODIFIE",
    "TITRE_MODIFIE", "COMMENTAIRE_ENSEIGNANT_LLM"
]

COLONNES_HITL_LLM = [
    "ID_GROUPE", "PARCOURS", "NIVEAU", "LACUNE_PRINCIPALE", "PREREQUIS",
    "CANDIDAT_LLM_ID", "TYPE_RESSOURCE_PROPOSE", "TITRE_PROPOSE",
    "LACUNE_COUVERTE", "JUSTIFICATION_LLM", "MODELE_LLM"
] + COLONNES_DECISION_LLM


def lire_csv_enseignant(chemin):
    """Lit un CSV édité sous Excel (séparateur , ou ; ; UTF-8 ou ANSI)."""

    derniere = None

    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(
                chemin, sep=None, engine="python",
                encoding=enc, dtype=str
            ).fillna("")
        except UnicodeDecodeError as e:
            derniere = e

    raise derniere


def archiver_fichier(chemin):

    horodatage = datetime.now().strftime("%Y%m%d_%H%M%S")
    cible = chemin.with_name(f"{chemin.stem}_ARCHIVE_{horodatage}{chemin.suffix}")
    chemin.rename(cible)

    return cible


def contient_decisions_llm(df):

    return (
        "DECISION_ENSEIGNANT_LLM" in df.columns
        and df["DECISION_ENSEIGNANT_LLM"].fillna("").astype(str).str.strip().ne("").any()
    )


def signature_llm(df):

    cols = ["ID_GROUPE", "CANDIDAT_LLM_ID", "TITRE_PROPOSE"]

    if df.empty or not all(c in df.columns for c in cols):
        return pd.DataFrame(columns=cols)

    return (
        df[cols].fillna("").astype(str)
        .sort_values(["ID_GROUPE", "CANDIDAT_LLM_ID"])
        .reset_index(drop=True)
    )


VALIDATION_LLM_TEMPLATE = PROPOSITIONS_LLM_A_VALIDER.copy()

for col in COLONNES_DECISION_LLM:
    VALIDATION_LLM_TEMPLATE[col] = ""

VALIDATION_LLM_TEMPLATE = VALIDATION_LLM_TEMPLATE[COLONNES_HITL_LLM]


def ecrire_template():
    VALIDATION_LLM_TEMPLATE.to_csv(FICHIER_HITL_LLM, index=False, encoding="utf-8-sig")


if VALIDATION_LLM_TEMPLATE.empty:

    # Aucun candidat : un ancien fichier ne doit pas rester en place,
    # sinon la cellule 27 validerait des décisions périmées.
    if FICHIER_HITL_LLM.exists():
        cible = archiver_fichier(FICHIER_HITL_LLM)
        print("Ancien fichier HITL LLM périmé archivé :", cible.name)

    print("Aucune proposition LLM à faire valider (générer d'abord avec la cellule 25B).")

elif FICHIER_HITL_LLM.exists():

    existant = lire_csv_enseignant(FICHIER_HITL_LLM)

    meme_generation = signature_llm(existant).equals(
        signature_llm(VALIDATION_LLM_TEMPLATE)
    )

    if contient_decisions_llm(existant) and meme_generation:
        print("Validation LLM existante protégée : aucun écrasement.")

    elif contient_decisions_llm(existant):
        cible = archiver_fichier(FICHIER_HITL_LLM)
        print(
            "Les candidats LLM ont changé : l'ancien fichier (avec décisions) "
            f"est archivé -> {cible.name}"
        )
        ecrire_template()
        print("Nouveau template HITL LLM créé : l'enseignant doit le remplir.")

    else:
        ecrire_template()
        print("Template HITL LLM mis à jour.")

else:

    ecrire_template()
    print("Fichier HITL LLM créé.")


print(FICHIER_HITL_LLM)

Les candidats LLM ont changé : l'ancien fichier (avec décisions) est archivé -> validation_enseignant_llm_ARCHIVE_20261010_170649.csv
Nouveau template HITL LLM créé : l'enseignant doit le remplir.
/home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/06_RAG_LLM_HITL/hitl/validation_enseignant_llm.csv


Cellule 27 — Recharger la validation des propositions LLM

In [37]:
# ============================================================
# CELLULE 27 — Rechargement de la validation enseignant LLM
# ============================================================

RESSOURCES_LLM_VALIDEES = pd.DataFrame()


if BESOINS_LLM.empty:

    VALIDATION_LLM_TERMINEE = True

    print(
        "Aucun besoin LLM."
    )


else:

    if not FICHIER_HITL_LLM.exists():

        raise FileNotFoundError(
            f"Validation enseignant LLM absente : "
            f"{FICHIER_HITL_LLM}"
        )


    HITL_LLM = pd.read_csv(
        FICHIER_HITL_LLM
    )


    for col in [
        "DECISION_ENSEIGNANT_LLM",
        "TYPE_RESSOURCE_MODIFIE",
        "TITRE_MODIFIE",
        "COMMENTAIRE_ENSEIGNANT_LLM"
    ]:

        if col not in HITL_LLM.columns:
            HITL_LLM[col] = ""

        HITL_LLM[col] = (
            HITL_LLM[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )


    HITL_LLM[
        "DECISION_ENSEIGNANT_LLM"
    ] = (
        HITL_LLM[
            "DECISION_ENSEIGNANT_LLM"
        ]
        .str.upper()
    )


    DECISIONS_AUTORISEES = {
        "",
        "VALIDE",
        "MODIFIE",
        "REJETE"
    }


    decisions_inconnues = (
        set(
            HITL_LLM[
                "DECISION_ENSEIGNANT_LLM"
            ].unique()
        )
        -
        DECISIONS_AUTORISEES
    )


    if decisions_inconnues:

        raise ValueError(
            "Décisions LLM inconnues : "
            f"{decisions_inconnues}"
        )


    en_attente = (
        HITL_LLM[
            HITL_LLM[
                "DECISION_ENSEIGNANT_LLM"
            ] == ""
        ]
    )


    if not en_attente.empty:

        display(
            en_attente[
                [
                    "ID_GROUPE",
                    "CANDIDAT_LLM_ID",
                    "TITRE_PROPOSE"
                ]
            ]
        )

        raise RuntimeError(
            "Validation LLM non terminée."
        )


    VALIDATION_LLM_TERMINEE = True


    lignes_validees = []


    for _, ligne in (
        HITL_LLM
        .iterrows()
    ):

        decision = (
            ligne[
                "DECISION_ENSEIGNANT_LLM"
            ]
        )


        if decision == "REJETE":

            continue


        if decision == "VALIDE":

            type_final = (
                ligne[
                    "TYPE_RESSOURCE_PROPOSE"
                ]
            )

            titre_final = (
                ligne[
                    "TITRE_PROPOSE"
                ]
            )


        elif decision == "MODIFIE":

            type_final = (
                ligne[
                    "TYPE_RESSOURCE_MODIFIE"
                ]
                .strip()
                .lower()
            )

            titre_final = (
                ligne[
                    "TITRE_MODIFIE"
                ]
                .strip()
            )


            if (
                type_final
                not in {
                    "lecon",
                    "exercice",
                    "quiz"
                }
            ):

                raise ValueError(
                    "MODIFIE : type final invalide pour "
                    f"{ligne['CANDIDAT_LLM_ID']}"
                )


            if titre_final == "":

                raise ValueError(
                    "MODIFIE : titre final manquant pour "
                    f"{ligne['CANDIDAT_LLM_ID']}"
                )


        lignes_validees.append(
            {
                "ID_GROUPE":
                    ligne["ID_GROUPE"],

                "PARCOURS":
                    ligne["PARCOURS"],

                "RESOURCE_ID":
                    ligne[
                        "CANDIDAT_LLM_ID"
                    ],

                "TITRE":
                    titre_final,

                "TYPE_RESSOURCE":
                    str(
                        type_final
                    )
                    .strip()
                    .lower(),

                "CHAPITRE":
                    "",

                "LECON":
                    "",

                "ORIGINE":
                    "LLM_VALIDE_ENSEIGNANT",

                "SOURCE_DECISION":
                    decision,

                "LACUNE_COUVERTE":
                    ligne[
                        "LACUNE_COUVERTE"
                    ]
            }
        )


    RESSOURCES_LLM_VALIDEES = (
        pd.DataFrame(
            lignes_validees
        )
    )


    groupes_llm = set(
        BESOINS_LLM[
            "ID_GROUPE"
        ]
    )


    groupes_couverts_llm = set(
        RESSOURCES_LLM_VALIDEES[
            "ID_GROUPE"
        ]
    ) if not RESSOURCES_LLM_VALIDEES.empty else set()


    GROUPES_NON_COUVERTS_APRES_LLM = (
        groupes_llm
        -
        groupes_couverts_llm
    )


    print(
        "Validation LLM terminée :",
        VALIDATION_LLM_TERMINEE
    )

    print(
        "Ressources LLM validées/modifiées :",
        len(
            RESSOURCES_LLM_VALIDEES
        )
    )

    print(
        "Groupes encore non couverts :",
        sorted(
            GROUPES_NON_COUVERTS_APRES_LLM
        )
    )


    display(
        HITL_LLM[
            "DECISION_ENSEIGNANT_LLM"
        ]
        .value_counts()
        .rename("N")
        .to_frame()
    )

Validation LLM terminée : True
Ressources LLM validées/modifiées : 6
Groupes encore non couverts : []


,N
DECISION_ENSEIGNANT_LLM,
VALIDE,3
MODIFIE,3


Cellule 28 — Ressources finales

In [38]:
# ============================================================
# CELLULE 28 — Consolidation RAG + LLM validé
# ============================================================

colonnes_finales = [
    "ID_GROUPE",
    "PARCOURS",
    "RESOURCE_ID",
    "TITRE",
    "TYPE_RESSOURCE",
    "CHAPITRE",
    "LECON",
    "ORIGINE",
    "SOURCE_DECISION"
]


rag_final = (
    RESSOURCES_RAG_VALIDEES[
        colonnes_finales
    ]
    .copy()
    if not RESSOURCES_RAG_VALIDEES.empty
    else
    pd.DataFrame(
        columns=colonnes_finales
    )
)


llm_final = (
    RESSOURCES_LLM_VALIDEES[
        colonnes_finales
    ]
    .copy()
    if not RESSOURCES_LLM_VALIDEES.empty
    else
    pd.DataFrame(
        columns=colonnes_finales
    )
)


RESSOURCES_FINALES = (
    pd.concat(
        [
            rag_final,
            llm_final
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=[
            "ID_GROUPE",
            "RESOURCE_ID"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Statut final de chaque groupe
# ------------------------------------------------------------

STATUT_FINAL_GROUPES = (
    STATUT_RAG_GROUPES[
        [
            "ID_GROUPE"
        ]
    ]
    .drop_duplicates()
    .copy()
)


compte_final = (
    RESSOURCES_FINALES
    .groupby(
        "ID_GROUPE"
    )
    .size()
)


STATUT_FINAL_GROUPES[
    "N_RESSOURCES_FINALES"
] = (
    STATUT_FINAL_GROUPES[
        "ID_GROUPE"
    ]
    .map(
        compte_final
    )
    .fillna(0)
    .astype(int)
)


STATUT_FINAL_GROUPES[
    "STATUT_FINAL"
] = np.where(
    STATUT_FINAL_GROUPES[
        "N_RESSOURCES_FINALES"
    ] > 0,

    "COUVERT",

    "NON_COUVERT_APRES_RAG_LLM"
)


print(
    "Ressources finales :",
    len(
        RESSOURCES_FINALES
    )
)


display(
    RESSOURCES_FINALES
)


display(
    STATUT_FINAL_GROUPES
)

Ressources finales : 25


,ID_GROUPE,PARCOURS,RESOURCE_ID,TITRE,TYPE_RESSOURCE,CHAPITRE,LECON,ORIGINE,SOURCE_DECISION
0,G001,BI,BI-C2-L2-LEC,Schéma en étoile et en flocon,lecon,C2 - Modélisation dimensionnelle,L2.2,KNOWLEDGE_RAG,VALIDE
1,G001,BI,BI-C2-L2-EXO,Exercice - Schéma en étoile et en flocon,exercice,C2 - Modélisation dimensionnelle,L2.2,KNOWLEDGE_RAG,VALIDE
2,G002,BI,BI-C3-L3-LEC,Qualité des données,lecon,C3 - ETL et entrepôts de données,L3.3,KNOWLEDGE_RAG,VALIDE
3,G002,BI,BI-C3-L3-EXO,Exercice - Qualité des données,exercice,C3 - ETL et entrepôts de données,L3.3,KNOWLEDGE_RAG,VALIDE
4,G003,BI,BI-C2-L2-LEC,Schéma en étoile et en flocon,lecon,C2 - Modélisation dimensionnelle,L2.2,KNOWLEDGE_REMPLACEMENT,REMPLACE
5,G003,BI,BI-C5-L1-LEC,Connexion et modélisation des données,lecon,C5 - Outils de reporting (Power BI / Tableau),L5.1,KNOWLEDGE_RAG,VALIDE
6,G005,DA,DA-C4-L3-LEC,Agrégation et fusion de données,lecon,C4 - Python et Pandas pour l'analyse,L4.3,KNOWLEDGE_RAG,VALIDE
7,G005,DA,DA-C4-L3-EXO,Exercice - Agrégation et fusion de données,exercice,C4 - Python et Pandas pour l'analyse,L4.3,KNOWLEDGE_RAG,VALIDE
8,G006,DA,DA-C2-L3-LEC,Nettoyage et mise en forme des données,lecon,C2 - Excel pour l'analyse,L2.3,KNOWLEDGE_RAG,VALIDE
9,G006,DA,DA-C2-L3-EXO,Exercice - Nettoyage et mise en forme des données,exercice,C2 - Excel pour l'analyse,L2.3,KNOWLEDGE_RAG,VALIDE


,ID_GROUPE,N_RESSOURCES_FINALES,STATUT_FINAL
0,G001,2,COUVERT
1,G002,2,COUVERT
2,G003,2,COUVERT
3,G004,3,COUVERT
4,G005,2,COUVERT
5,G006,2,COUVERT
6,G007,3,COUVERT
7,G008,2,COUVERT
8,G009,2,COUVERT
9,G010,1,COUVERT


Cellule 29 — Candidats validés pouvant être ajoutés au Knowledge Base

In [39]:
# ============================================================
# CELLULE 29 — Staging Knowledge Base
# ============================================================

COLONNES_KNOWLEDGE = [
    "RESOURCE_ID",
    "PARCOURS",
    "CHAPITRE",
    "LECON",
    "TITRE",
    "TYPE_RESSOURCE",
    "CONTENU",
    "SOURCE"
]


if RESSOURCES_LLM_VALIDEES.empty:

    CANDIDATS_KNOWLEDGE = (
        pd.DataFrame(
            columns=COLONNES_KNOWLEDGE
        )
    )


else:

    CANDIDATS_KNOWLEDGE = (
        RESSOURCES_LLM_VALIDEES[
            [
                "RESOURCE_ID",
                "PARCOURS",
                "CHAPITRE",
                "LECON",
                "TITRE",
                "TYPE_RESSOURCE"
            ]
        ]
        .copy()
    )


    CANDIDATS_KNOWLEDGE[
        "CONTENU"
    ] = ""


    CANDIDATS_KNOWLEDGE[
        "SOURCE"
    ] = (
        "llm_titre_valide_enseignant"
    )


    # --------------------------------------------------------
    # Sécurité anti-duplication avec le Knowledge actuel
    # --------------------------------------------------------

    titres_actuels = set(
        RESSOURCES[
            "TITRE"
        ]
        .dropna()
        .map(
            normaliser_titre
        )
    )


    doublons = (
        CANDIDATS_KNOWLEDGE[
            CANDIDATS_KNOWLEDGE[
                "TITRE"
            ]
            .map(
                normaliser_titre
            )
            .isin(
                titres_actuels
            )
        ]
    )


    if not doublons.empty:

        display(
            doublons
        )

        raise RuntimeError(
            "Un titre LLM validé existe déjà "
            "dans le Knowledge Base."
        )


    CANDIDATS_KNOWLEDGE = (
        CANDIDATS_KNOWLEDGE[
            COLONNES_KNOWLEDGE
        ]
    )


FICHIER_STAGING_KNOWLEDGE = (
    KNOWLEDGE_STAGING_DIR
    / "candidats_llm_valides_pour_knowledge.csv"
)


CANDIDATS_KNOWLEDGE.to_csv(
    FICHIER_STAGING_KNOWLEDGE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Candidats LLM validés prêts pour Knowledge :",
    len(
        CANDIDATS_KNOWLEDGE
    )
)


print(
    "AUCUNE écriture automatique dans course_resources.csv."
)


print(
    FICHIER_STAGING_KNOWLEDGE
)

Candidats LLM validés prêts pour Knowledge : 6
AUCUNE écriture automatique dans course_resources.csv.
/home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/06_RAG_LLM_HITL/knowledge_staging/candidats_llm_valides_pour_knowledge.csv


In [40]:
# ============================================================
# CELLULE 30 — Exports finaux Notebook 06
# ============================================================

RESSOURCES_RAG_VALIDEES.to_csv(
    TABLE_DIR
    / "ressources_rag_validees.csv",
    index=False,
    encoding="utf-8-sig"
)


BESOINS_LLM.to_csv(
    TABLE_DIR
    / "groupes_non_couverts_par_rag.csv",
    index=False,
    encoding="utf-8-sig"
)


RESSOURCES_LLM_VALIDEES.to_csv(
    TABLE_DIR
    / "ressources_llm_validees.csv",
    index=False,
    encoding="utf-8-sig"
)


RESSOURCES_FINALES.to_csv(
    TABLE_DIR
    / "ressources_finales.csv",
    index=False,
    encoding="utf-8-sig"
)


STATUT_FINAL_GROUPES.to_csv(
    TABLE_DIR
    / "statut_final_groupes.csv",
    index=False,
    encoding="utf-8-sig"
)


if not METRIQUES_RAG_GROUPES.empty:

    METRIQUES_RAG_GROUPES.to_csv(
        TABLE_DIR
        / "metriques_rag_groupes.csv",
        index=False,
        encoding="utf-8-sig"
    )


if not METRIQUES_RAG_GLOBALES.empty:

    METRIQUES_RAG_GLOBALES.to_csv(
        TABLE_DIR
        / "metriques_rag_globales.csv",
        index=False,
        encoding="utf-8-sig"
    )


print(
    "=" * 70
)

print(
    "NOTEBOOK 06 — ETAT FINAL"
)

print(
    "=" * 70
)

print(
    "Validation RAG terminée :",
    VALIDATION_RAG_TERMINEE
)

print(
    "Ressources RAG validées :",
    len(
        RESSOURCES_RAG_VALIDEES
    )
)

print(
    "Groupes envoyés au LLM :",
    len(
        BESOINS_LLM
    )
)

print(
    "Validation LLM terminée :",
    VALIDATION_LLM_TERMINEE
)

print(
    "Ressources LLM validées :",
    len(
        RESSOURCES_LLM_VALIDEES
    )
)

print(
    "Ressources finales :",
    len(
        RESSOURCES_FINALES
    )
)

print(
    "Candidats prêts pour Knowledge :",
    len(
        CANDIDATS_KNOWLEDGE
    )
)

print(
    "=" * 70
)

NOTEBOOK 06 — ETAT FINAL
Validation RAG terminée : True
Ressources RAG validées : 19
Groupes envoyés au LLM : 2
Validation LLM terminée : True
Ressources LLM validées : 6
Ressources finales : 25
Candidats prêts pour Knowledge : 6
